In [ ]:
import json
import pandas as pd

In [ ]:
# chuyển cột date sang kiểu datetime
def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        return pd.DataFrame(data)

df_price = load_json("../data/processed/price_history_processed.json")
df_price["date"] = pd.to_datetime(df_price["date"], dayfirst=True)
df_price["date"] = df_price["date"].dt.strftime("%Y-%m-%d")
df_price = df_price[(df_price['date'] >= '2024-01-01') & (df_price['date'] <= '2025-12-31')]
df_price = df_price.sort_values(['symbol', 'date'])





Phản ứng thị trường (01/06/2025 - 07/10/2025)

In [ ]:


# Lọc dữ liệu trong khoảng thời gian
mask = (df_price['date'] >= '2025-06-01') & (df_price['date'] <= '2025-10-07')
df_price_phase1 = df_price.loc[mask]
growth = df_price_phase1.groupby('symbol')['price_change_pct'].sum()
print(growth)
growth.to_csv('q1_growth.csv')

Biến động sau tin FTSE Russell (07/10/2025)

In [ ]:
daily_returns = df_price.copy()

# tạo cột Period (CASE WHEN)
daily_returns["Period"] = daily_returns["date"].apply(
    lambda x: "Before_News" if x <= "2025-10-07" else "After_News"
)

# lọc ngày (WHERE)
daily_returns = daily_returns[
    (daily_returns["date"] >= "2025-06-01") &
    (daily_returns["date"] <= "2025-12-31")
]

# GROUP BY + AVG
result = (
    daily_returns
    .groupby(["symbol", "Period"])
    .agg(
        Avg_Volume=("volume", "mean"),
        Avg_Daily_Return=("price_change_pct", "mean")
    )
    .reset_index()
)

print(result)
result.to_csv('q2_result.csv', index=False)

Hiệu ứng Momentum (đà tăng trưởng) của nhóm ngân hàng có thực
sự vượt trội trong các giai đoạn tin tức hay không?

In [ ]:
# Tách riêng VNIndex và nhóm Bank
bankId = [
        "ACB", "BID", "CTG", "EIB","EVF", "HDB", "KLB", "LPB", "MBB", "MSB", "NAB", 
        "OCB", "SHB", "SSB", "STB", "TCB", "TPB", "VAB", "VCB", "VIB", "VPB"
        ]
df_price['daily_return'] = df_price.groupby('symbol')['close'].pct_change()

mask_momentum = (df_price['date'] >= '2025-06-01') & (df_price['date'] <= '2025-12-31')
df_price_momentum = df_price.loc[mask_momentum]
# Tính return trung bình của nhóm Bank theo ngày
bank_return = df_price_momentum[df_price_momentum['symbol'].isin(bankId)].groupby('date')['price_change_pct'].mean()
vni_return = df_price_momentum[df_price_momentum['symbol'] == 'VNINDEX'].set_index('date')['price_change_pct']
excess_return = bank_return - vni_return

# Gộp lại để so sánh
momentum_df_price = pd.DataFrame({'Bank_Momentum': bank_return, 'VNIndex_Momentum': vni_return, 'Excess Return': excess_return}).dropna()
print(momentum_df_price)
momentum_df_price.to_csv('q3_momentum.csv')

Mức độ rủi ro thị trường (độ biến động, mức giảm tối đa,...)

In [ ]:
# 1. Volatility (Độ biến động năm)

volatility = df_price.groupby('symbol')['price_change_pct'].std() * (250**0.5)
# 1. Chuyển % về số thập phân

df_price['daily_ret'] = df_price['price_change_pct'] / 100

# 2. Tính Chỉ số tích lũy (Equity Curve) cho từng mã
# Giả sử bắt đầu với 1 đồng, ta dùng cumprod()
df_price['equity_curve'] = df_price.groupby('symbol')['daily_ret'].transform(lambda x: (1 + x).cumprod())

# 3. Tính Đỉnh (Running Peak)
df_price['peak'] = df_price.groupby('symbol')['equity_curve'].transform(lambda x: x.cummax())

# 4. Tính Drawdown theo từng thời điểm
df_price['drawdown'] = (df_price['equity_curve'] - df_price['peak']) / df_price['peak']

# 5. Lấy Max Drawdown (giá trị nhỏ nhất)
max_dd = df_price.groupby('symbol')['drawdown'].min() * 100
risk_metrics = pd.DataFrame({'Volatility': volatility, 'Max_Drawdown': max_dd})
print("--- Max Drawdown chuẩn (đã điều chỉnh cổ tức/chia tách) ---")
print(risk_metrics)
risk_metrics.to_csv('q4_risk.csv')

In [ ]:
delta = df_price['price_change_pct']

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df_price['RSI'] = 100 - (100/(1+rs))
print(df_price[('symbol', 'RSI')])